# Tool-call model training on TPU
Train the tool-call pointer model through its 4 curriculum stages: single easy calls, single calls among harder confusers, independent multi-call turns, then calls that need an earlier result. Each stage warm-starts from the last. Set `RUN_MODE` to `smoke` for 10 updates per stage or `full` for every epoch. Rerun the training cell after a session ends to resume the unfinished stage.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

RUN_MODE = 'smoke'  # 'smoke' or 'full'.
STAGES = (0, 1, 2, 3)  # Curriculum stages, in order; each warm-starts from the one before.
DEVICES = None  # Detect all visible TPUs; set 1 or 8 to require that exact count.
ROWS_PER_DEVICE = 4  # Sequences per TPU chip per microbatch.
PACK = True  # Pack several whole requests into each row; False keeps one request per row.
QUESTIONS_PER_ROW = 32  # Question slots per packed row.
PREFETCH = 2  # Updates prepared ahead on a background thread; 0 prepares inline.
SMOKE_STEPS = 10  # Updates per stage in smoke mode.
EPOCHS = 1  # Per stage.
SESSION_HOURS = 1 if RUN_MODE == 'smoke' else 8  # One deadline for every stage; rerun to resume.
MICROBATCHES = 1
SEQUENCE_TOKENS = None  # Joint question-and-text tokens per request; None measures every split.
QUESTIONS = None  # Questions per request; None measures every split.
OPTIMIZER = 'optax'
KEEP_CHECKPOINTS = 2  # Newest checkpoints kept per stage.
LEARNING_RATE = 2e-5  # Peak rate.
WARMUP_UPDATES = 500  # Each stage warms up again, then cosine-decays to FINAL_LR_FRACTION.
FINAL_LR_FRACTION = 0.5
ATTENTION = 'splash'  # 'splash' streams attention on TPU; 'dense' is the XLA reference.
LOCAL_WINDOW = None
GLOBAL_EVERY = 3
QUANTIZER = 'nf4'  # Also train an NF4 student on the same weights; None trains dense only.
DISTILL_WEIGHT = 1.0
TEMPERATURE = 2.0
VALIDATION_RECORDS = 8 if RUN_MODE == 'smoke' else 256
FINAL_RECORDS = 8 if RUN_MODE == 'smoke' else 1000  # Per source and question type in each split; 0 evaluates every record.

HOST = Path('/content')
SCRATCH = HOST / 'toolcalls-scratch'
OUTPUT = None  # Defaults to toolcalls-{RUN_MODE}-{DEVICES}dev under HOST; one folder per stage inside.
DATASET = None  # Override with a local copy of the dataset (stage0/ ... stage3/) to skip its download.
DATASET_REPO = 'protodotdesign/toolcalls-v1'
DATASET_REVISION = '9d63f7b434c26c2dadcead30a3152b55f9c298b1'
COPY_TO = None  # e.g. '/content/drive/MyDrive/toolcall-runs': copy results and bundles there at the end.

if RUN_MODE not in ('smoke', 'full'):
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
for name in ('ROWS_PER_DEVICE', 'MICROBATCHES', 'QUESTIONS_PER_ROW', 'SMOKE_STEPS', 'EPOCHS'):
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f'{name} must be a positive integer')
if not STAGES or any(stage not in (0, 1, 2, 3) for stage in STAGES):
    raise ValueError('STAGES must list stages 0 to 3')
if VALIDATION_RECORDS < 1 or FINAL_RECORDS < 0:
    raise ValueError('Invalid evaluation bound')
SCRATCH.mkdir(parents=True, exist_ok=True)
if COPY_TO and str(COPY_TO).startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive  # Mount now, while someone is here to approve it.
    drive.mount('/content/drive')
CHECKOUT = SCRATCH / 'minifield-training'
SOURCE_REPO_URL = 'https://github.com/Minifield-Labs/minifield-training.git'
SOURCE_REVISION = 'e838b616fd48385cb2bc4730a3a3c52770b3df9a'
MODEL = SCRATCH / 'encoder'
CACHE = SCRATCH / 'arrow'
SEED = 20261003

def setup_command(*command, cwd=None):
    result = subprocess.run(command, cwd=cwd or CHECKOUT, check=True,
                            text=True, stdout=subprocess.PIPE)
    print(result.stdout, end='', flush=True)
    return result.stdout


## Fetch the pinned source


In [ ]:
if not (CHECKOUT / '.git').is_dir():
    if CHECKOUT.exists():
        raise RuntimeError(f'Expected a clean checkout path: {CHECKOUT}')
    setup_command('git', 'clone', '--no-checkout', SOURCE_REPO_URL, str(CHECKOUT),
              cwd=SCRATCH)
else:
    if setup_command('git', 'status', '--porcelain').strip():
        raise RuntimeError(f'Checkout contains local changes: {CHECKOUT}')
contains_pin = subprocess.run(
    ['git', 'cat-file', '-e', f'{SOURCE_REVISION}^{{commit}}'],
    cwd=CHECKOUT, capture_output=True, check=False).returncode == 0
if not contains_pin:
    setup_command('git', 'fetch', '--depth', '1', SOURCE_REPO_URL, SOURCE_REVISION)
setup_command('git', 'checkout', '--detach', SOURCE_REVISION)
actual_revision = setup_command('git', 'rev-parse', 'HEAD').strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError(f'Expected {SOURCE_REVISION}, got {actual_revision}')
if not (CHECKOUT / 'examples/toolcalls/train.py').is_file():
    raise RuntimeError('The pinned checkout is missing the tool-call trainer')
print('Training source:', SOURCE_REPO_URL, actual_revision)

## Install into this kernel


In [ ]:
if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Select a Python 3.12 or 3.13 notebook runtime')
if any(name in sys.modules for name in ('jax', 'jaxlib', 'minifield_training')):
    raise RuntimeError('Restart the notebook session before installing dependencies')
import importlib.metadata
import re
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'uv==0.11.30'])
locked = SCRATCH / 'notebook-requirements.txt'
subprocess.check_call([
    sys.executable, '-m', 'uv', 'export', '--project', str(CHECKOUT),
    '--locked', '--no-dev', '--all-extras', '--no-emit-project', '--no-hashes',
    '--format', 'requirements-txt', '--output-file', str(locked),
])
# Kaggle imports some packages (numpy) before the first cell. Replacing a loaded
# package on disk would need a restart, so keep what the kernel already loaded and
# let pip confirm those versions satisfy everything else.
WATCHED = ('numpy', 'scipy', 'pandas', 'pyarrow', 'transformers', 'tokenizers')
preloaded = {name: sys.modules[name].__version__ for name in WATCHED if name in sys.modules}
requirements = SCRATCH / 'notebook-requirements-kept.txt'
requirements.write_text(''.join(
    line + '\n' for line in locked.read_text().splitlines()
    if re.split(r'[=<>!~; ]', line.strip(), maxsplit=1)[0].lower() not in preloaded
))
constraints = SCRATCH / 'notebook-preloaded.txt'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in preloaded.items()))
print('Keeping preloaded packages:', preloaded or 'none', flush=True)
extra = [str(CHECKOUT), 'jax[tpu]==0.7.2']
pip = [sys.executable, '-m', 'pip', 'install']
if subprocess.run([*pip, '-r', str(requirements), '-c', str(constraints), *extra]).returncode:
    # Incompatible preloaded versions: install the lock so the restarted kernel loads it.
    subprocess.check_call([*pip, '-r', str(locked), *extra])
    raise RuntimeError('Preloaded packages were incompatible and have been replaced. Restart the session, then run from the first cell.')
for package in WATCHED:
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package):
        raise RuntimeError(f'{package} changed on disk. Restart the session, then run from the first cell.')
sys.path.insert(0, str(CHECKOUT))
print('Installed into notebook kernel:', sys.executable)

## Discover TPU devices


In [ ]:
import json
from pathlib import Path
import sys
import jax

devices = jax.devices()
if jax.__version__ != '0.7.2':
    raise RuntimeError('Expected pinned JAX 0.7.2')
if jax.process_count() != 1 or len(devices) != jax.local_device_count():
    raise RuntimeError('This notebook requires all TPU devices on one host')
if not devices or any(device.platform != 'tpu' for device in devices):
    raise RuntimeError('Select a TPU runtime')
hardware = {'devices': len(devices), 'kinds': [d.device_kind for d in devices]}

print(hardware, flush=True)
if DEVICES is None:
    DEVICES = hardware['devices']
if type(DEVICES) is not int or DEVICES < 1 or hardware['devices'] != DEVICES:
    raise ValueError(f"Requested {DEVICES} devices; runtime exposes {hardware['devices']}")
ROWS = DEVICES * ROWS_PER_DEVICE
if OUTPUT is None:
    OUTPUT = HOST / f'toolcalls-{RUN_MODE}-{DEVICES}dev'
OUTPUT = Path(OUTPUT)
print('Mode:', RUN_MODE, 'Devices:', DEVICES,
      'Rows per device:', ROWS_PER_DEVICE, 'Packing:', PACK,
      'Rows per update:', ROWS * MICROBATCHES, 'Output:', OUTPUT)

## Download the curriculum and encoder


In [ ]:
from huggingface_hub import snapshot_download
from minifield_training.models.lfm2_5 import encoder

if DATASET is None:
    DATASET = SCRATCH / 'dataset'
    snapshot_download(DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION,
                      local_dir=str(DATASET), allow_patterns=[
                          f'stage{stage}/{pattern}' for stage in STAGES for pattern in (
                              'manifest.json', 'identity.json', 'report.json', 'sources.json',
                              'tokenizer/*', 'train/*.parquet', 'validation/*.parquet',
                              'calibration/*.parquet', 'test/*.parquet', 'ood/*.parquet')])
DATASET = Path(DATASET)
for stage in STAGES:
    manifest = json.loads((DATASET / f'stage{stage}' / 'manifest.json').read_text())
    assert manifest['format'] == 'minifield.magicbox/1.0' and manifest['complete'] is True
    by_split = {}
    for shard in manifest['shards']:
        by_split[shard['split']] = by_split.get(shard['split'], 0) + shard['rows']
    print(f'Stage {stage} records:', by_split)
snapshot_download(encoder.SOURCE.model_id, revision=encoder.SOURCE.revision,
                  allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'],
                  local_dir=str(MODEL))


## Train the curriculum
Each stage writes checkpoints, metrics, a bundle and its final evaluation under `OUTPUT/stage{N}`. The last stage also exports trimmed device bundles.


In [ ]:
from examples.magicbox import train
from examples.toolcalls import train as toolcalls
from minifield_training.supervisor import diagnostics

def report(event):
    line = json.dumps(event)
    print(line, flush=True)
    with (OUTPUT / 'progress.jsonl').open('a', encoding='utf-8') as log:
        log.write(line + '\n')

OUTPUT.mkdir(parents=True, exist_ok=True)
base = train.Settings(
    dataset=DATASET, model_dir=MODEL, cache=CACHE, devices=DEVICES, platform='tpu',
    rows=ROWS, microbatches=MICROBATCHES, pack=PACK, questions_per_row=QUESTIONS_PER_ROW,
    prefetch=PREFETCH, sequence_tokens=SEQUENCE_TOKENS, questions=QUESTIONS,
    epochs=EPOCHS, learning_rate=LEARNING_RATE, optimizer=OPTIMIZER, seed=SEED,
    warmup_updates=WARMUP_UPDATES, final_lr_fraction=FINAL_LR_FRACTION,
    attention=encoder.Attention(ATTENTION, LOCAL_WINDOW, GLOBAL_EVERY),
    quantizer=QUANTIZER, distill_weight=DISTILL_WEIGHT, temperature=TEMPERATURE,
)
with diagnostics.monitor(OUTPUT / 'diagnostics'):
    bundles = toolcalls.run_curriculum(
        base, DATASET, OUTPUT, stages=STAGES, session_seconds=SESSION_HOURS * 3600,
        validation_records=VALIDATION_RECORDS, final_records=FINAL_RECORDS,
        keep_checkpoints=KEEP_CHECKPOINTS,
        max_updates=SMOKE_STEPS if RUN_MODE == 'smoke' else None, report=report,
    )
print('Finished stages:', {stage: str(path) for stage, path in bundles.items()}, flush=True)
if len(bundles) < len(STAGES):
    print('The session deadline stopped the curriculum. Rerun this cell to resume.', flush=True)


## Copy results


In [ ]:
if COPY_TO:
    # Colab wipes /content when the runtime ends; checkpoints stay behind.
    import shutil
    destination = Path(COPY_TO) / OUTPUT.name
    destination.mkdir(parents=True, exist_ok=True)
    if (OUTPUT / 'progress.jsonl').is_file():
        shutil.copy2(OUTPUT / 'progress.jsonl', destination / 'progress.jsonl')
    for stage in STAGES:
        stage_output = OUTPUT / f'stage{stage}'
        keep = [*stage_output.glob('bundle-*'), *stage_output.glob('device-*'),
                stage_output / 'metrics', stage_output / 'run.json', *stage_output.glob('final-*.json')]
        for path in keep:
            target = destination / stage_output.name / path.name
            if path.is_dir():
                shutil.copytree(path, target, dirs_exist_ok=True)
            elif path.is_file():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
    print('Copied results and bundles to', destination, flush=True)


## Reload the last bundle and predict


In [ ]:
from examples.magicbox import bundle as magicbox_bundle
from examples.magicbox import composition as magicbox
from examples.magicbox import data as magicbox_data
from examples.magicbox import tokenizer
from examples.toolcalls import data
from minifield_training.batching import pointer as batching
from minifield_training.evaluation import pointer as evaluate_pointer
from minifield_training.objectives import schema_fields as weighting

# Bundles hold the markers folded into their embedding rows, so the plain
# pointer forward runs them.
last = bundles[max(bundles)]
loaded_cfg, loaded_head, loaded_parameters, decode_config = magicbox_bundle.load_pointer(
    last, templates=(data.TEMPLATE,))
adapter = tokenizer.Adapter(last / 'tokenizer')
request = {
    'state': 'User: Restart the pod back-end-pod, it is stuck.',
    'questions': {
        'next_tool': {'type': 'choice', 'instructions': 'Which tool should be called next?',
                      'criteria': {'restart_pod': 'Restart a named pod.',
                                   'scale_deployment': 'Change how many replicas a deployment runs.',
                                   'none': 'No listed tool fits.'}},
        'pod_name': {'type': 'extract', 'instructions': 'Argument `pod_name`'},
    },
}
record = data.marked(magicbox_data.compile_pointer_record('inference', request, {}, adapter.encode),
                     request, adapter.encode)
predictor = evaluate_pointer.Predictor(
    magicbox.bind_pointer(loaded_cfg, loaded_head, bf16=True),
    batching.PointerBatchStrategy(
        batching.Shape(1, 1, record.sequence_tokens, len(record.questions), loaded_cfg.vocab_size, 0),
        weighting.balance_types),
    presence_threshold=float(str(decode_config['presence_threshold'])),
)
predictions, _ = predictor.score(loaded_parameters, record, include_losses=False)
print(json.dumps(magicbox_data.format_results(record, predictions), indent=2))
